In [ ]:
##install and import necessary modules
##this code was originally designed and run in google colab
##use outside of colab may require modification
##if using colab, you may need to restart your runtime after installing modules,
##depending on enviornment at time of code running.

!pip install scikit-learn==1.5.2
!pip install tensorflow==2.12.1
!pip install xgboost==2.0.2
!pip install shap
!pip install statsmodels
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import seaborn as sn
import sys
import sklearn
import statsmodels.api as sm
from google.colab import drive
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils import resample
from IPython import display
from pandas.api.types import is_numeric_dtype
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve, recall_score, confusion_matrix, brier_score_loss, f1_score

sn.set(style='whitegrid')
pd.set_option('display.max_columns', None)

print("Python version:", sys.version)
print("scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgb.__version__)
print("shap version:", shap.__version__)

In [ ]:
# #import your dataset
# ##mount google drive if using in colab. Replace <MOUNT_POINT> with the directory where you want to mount the drive (e.g., /content/drive).
drive.mount('<MOUNT_POINT>')

# # Replace <YOUR_FILE_PATH> with the actual path inside your Google Drive (e.g., My Drive/FileNameHere).
file_path = '<MOUNT_POINT>/<YOUR_FILE_PATH>.csv'

In [ ]:
##specify columns to load from your dataset.  We will only load the columns necessary for the score and the variables necessary for the ISS/TRISS comparisons

columns_to_load = ['AGEYEARS', 'TOTALGCS', 'SBP'
                  , 'TEMPERATURE'
                  , 'PULSERATE', 'TRISS', 'TRISS_Death', 'MORTALITY', 'TRAUMATYPE'
                  , 'WEIGHT'
                  , 'ISS_05', 'NumberOfInjuriesOld', "FacilityKey",
                   'IntracranialVascularInjury','BrainStemInjury','EDH','SAH','SDH','SkullFx','DAI','NeckVascularInjury','ThoracicVascularInjury','AeroDigestiveInjury',
                   'CardiacInjury','LungInjury','AbdominalVascular','RibFx','KidneyInjury','StomachInjury','SpleenInjury','UroGenInternalInjury','SCI','SpineFx',
                   'UEAmputation','UEVascularInjury','UELongBoneFx','LEVascularInjury','PelvicFx','LEAmputation','PancreasInjury','LELongBoneFx','LiverInjury',
                   'ColorectalInjury','SmallBowelInjury','IPH', 'RESPIRATORYRATE', "SEX"
                   ]

columns_to_load_test = ['AGEYEARS', 'TOTALGCS', 'SBP'
                  , 'TEMPERATURE'
                  , 'PULSERATE', 'MORTALITY', 'TRAUMATYPE'
                  , 'WEIGHT'
                  , 'ISS_05', 'NumberOfInjuries',
                   'IntracranialVascularInjury','BrainStemInjury','EDH','SAH','SDH','SkullFx','DAI','NeckVascularInjury','ThoracicVascularInjury','AeroDigestiveInjury',
                   'CardiacInjury','LungInjury','AbdominalVascular','RibFx','KidneyInjury','StomachInjury','SpleenInjury','UroGenInternalInjury','SCI','SpineFx',
                   'UEAmputation','UEVascularInjury','UELongBoneFx','LEVascularInjury','PelvicFx','LEAmputation','PancreasInjury','LELongBoneFx','LiverInjury',
                   'ColorectalInjury','SmallBowelInjury','IPH', "RevisedTraumaScore", 'RespiratoryRate', 'SEX'
                   ]

In [ ]:
# Import data and specify missing values (both training and test sets)
data = pd.read_csv(file_path, na_values=['NA', 'N/A', 'NULL', ' ', '', '-99', '-98', '-99.0', '-99.00', '-98.0', '-98.00', 'NaN'], usecols=columns_to_load)
data_test = pd.read_csv(file_path_test, na_values=["*BL",'NA', 'N/A', 'NULL', ' ', '', '-99', '-98', '-99.0', '-99.00', '-98.0', '-98.00', 'NaN'], usecols=columns_to_load_test)


##ensure SBP and HR are of datatype numeric
for df in (data, data_test):
    for c in ['SBP', 'PULSERATE']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')


# Filter out rows where 'TRAUMATYPE' is not "Blunt" or "Penetrating"
try:
  exclude_values = ['26', 'Other/unspecified', 'Burn']
  data = data[~data['TRAUMATYPE'].isin(exclude_values)]
except:
  pass

##compute TRISS
b0_blunt=-1.2470
b0_pen=-0.6029
b1_blunt=0.9544
b1_pen=	1.1430
b2_blunt=-0.0768
b2_pen=-0.1516
b3_blunt=-1.9052
b3_pen=-2.6676

## Age indicator (A)
A = np.where(data_test['AGEYEARS'].isna(), np.nan,
             (data_test['AGEYEARS'] > 54).astype(int))

## choose coefficients based on trauma type
# --- Trauma type parsing: missing trauma -> NaN; unknown strings -> NaN ---
tt = data_test['TRAUMATYPE'].astype('string').str.strip().str.lower()

is_pen = tt.eq('penetrating') | tt.eq('pen')
is_blunt = tt.eq('blunt') | tt.eq('bl')

# anything not recognized becomes missing
valid = is_pen | is_blunt

b0 = np.where(~valid, np.nan, np.where(is_pen, b0_pen, b0_blunt))
b1 = np.where(~valid, np.nan, np.where(is_pen, b1_pen, b1_blunt))
b2 = np.where(~valid, np.nan, np.where(is_pen, b2_pen, b2_blunt))
b3 = np.where(~valid, np.nan, np.where(is_pen, b3_pen, b3_blunt))

## linear predictor
b = (
    b0
    + b1 * data_test['RevisedTraumaScore']
    + b2 * data_test['ISS_05']
    + b3 * A
)

## probability of survival
data_test['TRISS'] = (1 / (1 + np.exp(-b))).clip(0, 1)
data_test['TRISS_Death'] = (1 - data_test['TRISS']).clip(0, 1)

##explicitly list variables that need to be present for inclusion and drop cases without these
##we cannot compare our score to ISS/TRISS without those metrics, and we need our target outcome mortality
required_vars = ['ISS_05', 'TRISS_Death', 'MORTALITY', 'SBP', 'TOTALGCS']
data = data.dropna(subset=required_vars)
data_test = data_test.dropna(subset=required_vars)

##dropRTS from test set
data_test = data_test.drop(columns=['RevisedTraumaScore'])

# Create ShockIndex with the required logic
data['ShockIndex'] = np.where(
    data['SBP'] == 0, 2.0,  # Case where SBP is 0 → set ShockIndex to 2.0
    data['PULSERATE'] / data['SBP']  # Normal calculation
)
data_test['ShockIndex'] = np.where(
    data_test['SBP'] == 0, 2.0,  # Case where SBP is 0 → set ShockIndex to 2.0
    data_test['PULSERATE'] / data_test['SBP']  # Normal calculation
)

# Set ShockIndex to NaN if PULSERATE or SBP is missing
data.loc[data['PULSERATE'].isna() | data['SBP'].isna(), 'ShockIndex'] = np.nan
data_test.loc[data_test['PULSERATE'].isna() | data_test['SBP'].isna(), 'ShockIndex'] = np.nan

##reset indices of the df
data.reset_index(drop=True, inplace=True)
data_test.reset_index(drop=True, inplace=True)

In [ ]:
##Rename variable from dataset upload
data = data.rename(columns={'NumberOfInjuriesOld': 'NumberOfInjuries'})

In [ ]:
##Drop variables needed for fairness comparisons and/or for RTS calculations that aren't used int he model
data=data.drop(columns=['SEX', "RESPIRATORYRATE"])

In [ ]:
##Drop variables needed for fairness comparisons and/or for RTS calculations that aren't used int he model
data_test=data_test.drop(columns=['SEX', "RespiratoryRate"])

In [ ]:
##verify data appears as intended (training)
data.head()

In [ ]:
##verify data_test appears as intended (test)
data_test.head()

In [ ]:
##check for missing values (training)
data.isnull().sum(axis=0)

In [ ]:
##check for missing values (test)
data_test.isnull().sum(axis=0)

In [ ]:
##create a datafram of all complications/vars to remove later.  We can remove all of these from the X data set and pick one to be
#our Y dataset

complications_df=pd.DataFrame()
complications_list= [
                    'MORTALITY', 'TRISS'
                    ]
for c in complications_list:
    complications_df[c] = data[c]
complications_df

In [ ]:
##this is where we choose our outcome variable, mortality, and give it its own dataframe

Y_data = pd.DataFrame()
Y_data['MORTALITY'] = data['MORTALITY']
Y_data

In [ ]:
##now do the same thing for test set

Y_data_test = pd.DataFrame()
Y_data_test['MORTALITY'] = data_test['MORTALITY']
Y_data_test

In [ ]:
##clean Y_data by replacing "Yes" and "No" vcalues with 0's and 1's

Y_data['MORTALITY'] = Y_data['MORTALITY'].replace({'Yes': 1, 'No': 0})
Y_data

In [ ]:
##clean Y_data_test by replacing "Yes" and "No" vcalues with 0's and 1's

Y_data_test['MORTALITY'] = Y_data_test['MORTALITY'].replace({'Yes': 1, 'No': 0})
Y_data_test

In [ ]:
##now drop the outcome from our feature space as well as TRISS, since were using 1-TRISS (aka TRISS_Death) and this varibale is now useless
X_data = data.drop(columns=['MORTALITY', 'TRISS'])
X_data.shape

In [ ]:
##same thing for test set
X_data_test = data_test.drop(columns=['MORTALITY', 'TRISS'])
X_data_test.shape

In [ ]:
##ensure no missing outcome data
Missing_Y = Y_data.isnull().sum(axis=0)
Missing_Y

In [ ]:
##ensure no missing outcome data (test set)
Missing_Y_test = Y_data_test.isnull().sum(axis=0)
Missing_Y_test

In [ ]:
##If we have no missing values here, our data is clean (training)
Y_clean=Y_data
del Y_data

In [ ]:
##If we have no missing values here, our data is clean (test)
Y_clean_test=Y_data_test
del Y_data_test

In [ ]:
##if above check passes, outcome data is now clean
Missing_Y_clean = Y_clean.isnull().sum(axis=0)
Missing_Y_clean

In [ ]:
##if above check passes, outcome data is now clean (test)
Missing_Y_clean_test = Y_clean_test.isnull().sum(axis=0)
Missing_Y_clean_test

In [ ]:
##check which variables in the input space have missing variables (training)

Missing = X_data.isnull().sum(axis=0)
Missing[Missing>0]

In [ ]:
##check which variables in the input space have missing variables (test)

Missing_test = X_data_test.isnull().sum(axis=0)
Missing_test[Missing_test>0]

In [ ]:
##order variables with missing data by percentage (training)

data_missing = (X_data.isnull().sum(axis=0)/X_data.shape[0]) * 100
data_missing

In [ ]:
##order variables with missing data by percentage (test)

data_missing_test = (X_data_test.isnull().sum(axis=0)/X_data_test.shape[0]) * 100
data_missing_test

In [ ]:
##display variables withOUT mising data (training)

data_missing[data_missing == 0].index

In [ ]:
##display variables withOUT mising data (test)

data_missing_test[data_missing_test == 0].index

In [ ]:
#remove the good columns (no missing values) from data_missing

data_missing = data_missing.drop(data_missing[data_missing == 0].index)
data_missing

In [ ]:
#remove the good columns (no missing values) from data_missing_test

data_missing_test = data_missing_test.drop(data_missing_test[data_missing_test == 0].index)
data_missing_test

In [ ]:
#sort this in ascending order
data_missing = data_missing.sort_values(ascending=False)
data_missing

In [ ]:
#sort this in ascending order (test)
data_missing_test = data_missing_test.sort_values(ascending=False)
data_missing_test

In [ ]:
##prepare to drop variables with >50% missing values

dropCutoff=50
bad_column_names = data_missing[data_missing >=dropCutoff].index
bad_column_names

In [ ]:
##check which variables are above this >50% missingness cutoff

bad_column_names_test = data_missing_test[data_missing_test >=dropCutoff].index
bad_column_names_test

In [ ]:
##actually drop bad variables (training)
X_data_new=X_data.drop(columns=bad_column_names, axis=1)

##check for which variables still have missing data (<50% missing values)
Missing = X_data_new.isnull().sum(axis=0)
Missing[Missing>0]

In [ ]:
##actually drop bad variables (test)
X_data_new_test=X_data_test.drop(columns=bad_column_names, axis=1)

##check for which variables still have missing data (<50% missing values)
Missing_test = X_data_new_test.isnull().sum(axis=0)
Missing_test[Missing_test>0]

In [ ]:
#display columns with less than 50% missing that need to be cleaned (training)

to_be_cleaned_column_names = data_missing[data_missing <50].index
to_be_cleaned_column_names

In [ ]:
#display columns with less than 50% missing that need to be cleaned (test)

to_be_cleaned_column_names_test = data_missing_test[data_missing_test <50].index
to_be_cleaned_column_names_test

In [ ]:
# Rename the 'TRAUMATYPE' column to 'Penetrating' and map the values to 0 and 1
X_data_new['Penetrating'] = X_data_new['TRAUMATYPE'].map({'Penetrating': 1, 'Blunt': 0})

# Drop the old 'TRAUMATYPE' column
X_data_new.drop(columns=['TRAUMATYPE'], inplace=True)

print(X_data_new.head())

In [ ]:
#same thing for test set
# Rename the 'TRAUMATYPE' column to 'Penetrating' and map the values to 0 and 1
X_data_new_test['Penetrating'] = X_data_new_test['TRAUMATYPE'].map({'Penetrating': 1, 'Blunt': 0})

# Drop the old 'TRAUMATYPE' column
X_data_new_test.drop(columns=['TRAUMATYPE'], inplace=True)

print(X_data_new_test.head())

In [ ]:
# Display the entire DataFrame without truncation
pd.set_option('display.max_columns', None)

# Get column names and data types
columns_info = []
for column_name, dtype in zip(X_data_new.columns, X_data_new.dtypes):
    columns_info.append(f"{column_name}: {dtype}")

formatted_columns_info = "\n".join(columns_info)

# Print column names and data types
print("Column Names and Data Types:")
print(formatted_columns_info)

In [ ]:
# Display the entire DataFrame without truncation (now for test)
pd.set_option('display.max_columns', None)

# Get column names and data types
columns_info_test = []
for column_name, dtype in zip(X_data_new_test.columns, X_data_new_test.dtypes):
    columns_info_test.append(f"{column_name}: {dtype}")

formatted_columns_info_test = "\n".join(columns_info)

# Print column names and data types
print("Column Names and Data Types:")
print(formatted_columns_info_test)

In [ ]:
##convert No's and Yes's to 0's and 1's to minimize the amount of double variables (want to avoid Yes/Nos being converted to 1-hot variables)

try:
    X_data_new= X_data_new.replace({True: 1, 'Yes': 1, "Female": 1, False: 0, 'No': 0, "Male": 0})
except:
    pass

X_data_new.head()

In [ ]:
##convert No's and Yes's to 0's and 1's to minimize the amount of double variables (want to avoid Yes/Nos being converted to 1-hot variables) (test set)

try:
    X_data_new_test= X_data_new_test.replace({True: 1, 'Yes': 1, "Female": 1, False: 0, 'No': 0, "Male": 0})
except:
    pass

X_data_new_test.head()

In [ ]:
#now prepare to do train/test splitting at the facility level

# total number of unique facilities
num_facilities = data["FacilityKey"].nunique()
print(f"Total number of facilities: {num_facilities}")

# build table of facility case counts
facility_counts = (
    data["FacilityKey"]
    .value_counts()
    .reset_index(name="CaseCount")       # force the count column name
    .rename(columns={"index": "FacilityKey"})
    .sort_values("CaseCount", ascending=False)   # sort by size
    .reset_index(drop=True)              # clean index after sorting
)

# print all rows without truncation
with pd.option_context("display.max_rows", None):
    print(facility_counts)

# delete intermediate df
del facility_counts

In [ ]:
##verify correct data shape
X_data_new.shape

In [ ]:
##preppare for hospital level splitting with various helper functions

def _extract_y_vector(y, y_col):
    if isinstance(y, pd.Series):
        return y
    if isinstance(y, pd.DataFrame):
        return y[y_col] if y_col is not None else (
            y.iloc[:, 0] if y.shape[1] == 1 else
            (_ for _ in ()).throw(ValueError("Y_clean has multiple columns; provide y_col."))
        )
    raise TypeError("Y_clean must be a pandas Series or DataFrame.")

def make_hospital_splits_Xy_train_cal(
    X, Y,
    facility_col="FacilityKey",
    y_col=None,
    train_prop=0.80, cal_prop=0.20,
    mega_quantile=0.95,
    n_bins=10,
    handle_missing_facility="drop",  # 'drop' or 'assign_train'
    seed=42
):
    assert abs(train_prop + cal_prop - 1.0) < 1e-9, "Proportions must sum to 1."

    # ---- Align X and Y on common index ----
    common_idx = X.index.intersection(Y.index)
    if (len(common_idx) != len(X)) or (len(common_idx) != len(Y)):
        print(f"[info] Aligning X and Y to common index: {len(common_idx)} rows kept (X={len(X)}, Y={len(Y)}).")
    X = X.loc[common_idx].copy()
    Y = Y.loc[common_idx].copy()

    # ---- Facility column (from X if present else Y) ----
    if facility_col in X.columns:
        fac = X[facility_col].copy()
    elif facility_col in Y.columns:
        fac = Y[facility_col].copy()
    else:
        raise KeyError(f"'{facility_col}' not found in X_data_new or Y_clean.")
    fac.name = facility_col

    # ---- Handle missing FacilityKey rows ----
    n_missing = fac.isna().sum()
    if n_missing > 0:
        if handle_missing_facility == "drop":
            keep = fac.notna()
            print(f"[warn] Dropping {n_missing} rows with missing {facility_col}.")
            X, Y, fac = X.loc[keep], Y.loc[keep], fac.loc[keep]
        elif handle_missing_facility == "assign_train":
            fac = fac.fillna("__UNKNOWN__")
        else:
            raise ValueError("handle_missing_facility must be 'drop' or 'assign_train'.")

    # ---- Outcome vector (optional, for summaries) ----
    y_vec = None
    try:
        y_vec = _extract_y_vector(Y, y_col).loc[X.index]
    except Exception:
        pass

    # ---- Hospital-level table ----
    hosp = fac.value_counts(dropna=False).rename_axis(facility_col).rename("n").to_frame()
    if y_vec is not None:
        ev = y_vec.groupby(fac).sum()
        hosp["events"] = ev.reindex(hosp.index).fillna(0).astype(int)
    else:
        hosp["events"] = np.nan

    # ---- Identify & balance mega hospitals ----
    thr = hosp["n"].quantile(mega_quantile)
    mega_idx = hosp[hosp["n"] >= thr].sort_values("n", ascending=False).index
    rest_idx = hosp.index.difference(mega_idx)

    splits = {h: None for h in hosp.index}
    load = {"train": 0, "cal": 0}
    order = ["train", "cal"]

    for h in mega_idx:
        target = min(load, key=load.get)
        ties = [k for k, v in load.items() if v == load[target]]
        if len(ties) > 1:
            for name in order:
                if name in ties:
                    target = name
                    break
        splits[h] = target
        load[target] += int(hosp.loc[h, "n"])

    # ---- Stratify remaining by log-volume and sample ----
    rng = np.random.default_rng(seed)
    if len(rest_idx) > 0:
        rest = hosp.loc[rest_idx].copy()
        rest["log_n"] = np.log1p(rest["n"])
        rest["vol_bin"] = pd.qcut(rest["log_n"], q=n_bins, duplicates="drop")

        for _, sub in rest.groupby("vol_bin", observed=True):
            ids = sub.index.to_numpy()
            rng.shuffle(ids)
            k = len(ids)

            n_train = int(round(train_prop * k))
            n_cal   = k - n_train  # remainder to cal

            # small-bin safeguard: try to ensure both non-empty when possible
            if k >= 2:
                if n_train == 0:
                    n_train = 1
                    n_cal = k - n_train
                if n_cal == 0:
                    n_cal = 1
                    n_train = k - n_cal

            assign = {
                "train": ids[:n_train],
                "cal":   ids[n_train:]
            }
            for name, arr in assign.items():
                for h in arr:
                    splits[h] = name

    # ---- Fallback: assign any leftover hospitals (None) to smallest-load split ----
    leftover = [h for h, s in splits.items() if s is None]
    if leftover:
        print(f"[warn] {len(leftover)} hospital(s) were unassigned after stratification; assigning by load balance.")
        for h in leftover:
            target = min(load, key=load.get)
            splits[h] = target
            load[target] += int(hosp.loc[h, "n"])

    # ---- Build mapping and assign rows ----
    split_map = pd.DataFrame({facility_col: list(splits.keys()), "split": list(splits.values())})
    assert split_map["split"].isna().sum() == 0, "Some hospitals were not assigned (unexpected)."

    split_series = fac.map(split_map.set_index(facility_col)["split"])

    X_train = X[split_series.eq("train")].copy()
    X_cal   = X[split_series.eq("cal")].copy()

    Y_train = Y.loc[X_train.index].copy()
    Y_cal   = Y.loc[X_cal.index].copy()

    # ---- Summaries ----
    def _summary(ix, name):
        nrows = len(ix)
        uh = fac.loc[ix].nunique(dropna=False)
        if y_vec is not None:
            ev = int(y_vec.loc[ix].sum())
            print(f"{name:>5} | rows={nrows:,} | hospitals={uh:,} | events={ev:,}")
        else:
            print(f"{name:>5} | rows={nrows:,} | hospitals={uh:,}")

    print("\n=== Split summary (rows / hospitals / events) ===")
    _summary(X_train.index, "train")
    _summary(X_cal.index,   "cal")

    return X_train, X_cal, Y_train, Y_cal, split_map, hosp.sort_values("n", ascending=False)

In [ ]:
## split into train + calibration sets (facility-level)

X_train_cal, X_val_cal, Y_train_cal, Y_val_cal, split_map, hosp_table = make_hospital_splits_Xy_train_cal(
    X_data_new, Y_clean,
    facility_col="FacilityKey",
    y_col="MORTALITY",      # omit or set None if Y_clean is a Series already
    train_prop=0.80, cal_prop=0.20,
    mega_quantile=0.95, n_bins=10, seed=0
)

In [ ]:
##Lets delete some intermediate variablen assignments from memory to free up ram

X_test = X_data_new_test
Y_test = Y_clean_test

del X_data_new_test
del Y_clean_test
del data_test
del data

import gc
gc.collect()

In [ ]:
# Drop FacilityKey from X predictors
for df in (X_train_cal, X_val_cal):
    df.drop(columns=["FacilityKey"], inplace=True, errors="ignore")


In [ ]:
# before imputation, lets figure out which vars have missing data that need to be imputed
to_be_cleaned_column_names = [c for c in to_be_cleaned_column_names if c != 'FacilityKey']

In [ ]:
# before imputation (test set)
to_be_cleaned_column_names_test = [c for c in to_be_cleaned_column_names_test if c != 'FacilityKey']

In [ ]:
##now design helper function to do median/mode imputation

def impute_from_train(X_train_cal, X_val_cal, to_be_cleaned_column_names):
    for c in to_be_cleaned_column_names:
        s_train = X_train_cal[c]

        if is_numeric_dtype(s_train):
            median_value = s_train.median(skipna=True)
            if pd.isna(median_value):
                raise ValueError(f"All values missing in TRAIN for numeric column '{c}'; cannot compute median.")
            for df in (X_train_cal, X_val_cal):
                df.loc[:, c] = df[c].fillna(median_value)
        else:
            mode_series = s_train.mode(dropna=True)
            if mode_series.empty:
                raise ValueError(f"All values missing in TRAIN for categorical column '{c}'; cannot compute mode.")
            mode_value = mode_series.iloc[0]
            for df in (X_train_cal, X_val_cal):
                df.loc[:, c] = df[c].astype(object).fillna(mode_value)

# actually used function
impute_from_train(X_train_cal, X_val_cal, to_be_cleaned_column_names)

In [ ]:
##same thing as above for test set

def impute_from_train_test(X_train_cal, X_test, to_be_cleaned_column_names_test):
    for c in to_be_cleaned_column_names_test:
        s_train = X_train_cal[c]

        if is_numeric_dtype(s_train):
            median_value = s_train.median(skipna=True)
            if pd.isna(median_value):
                raise ValueError(f"All values missing in TRAIN for numeric column '{c}'; cannot compute median.")
            for df in (X_train_cal, X_test):
                df.loc[:, c] = df[c].fillna(median_value)
        else:
            mode_series = s_train.mode(dropna=True)
            if mode_series.empty:
                raise ValueError(f"All values missing in TRAIN for categorical column '{c}'; cannot compute mode.")
            mode_value = mode_series.iloc[0]
            for df in (X_train_cal, X_test):
                df.loc[:, c] = df[c].astype(object).fillna(mode_value)

# usage
impute_from_train_test(X_train_cal, X_test, to_be_cleaned_column_names_test)

In [ ]:
#recompute shock index after imputation to ensure no mismatched HR/SBP vars with SI var
def recompute_shock_index(df, pulse_col='PULSERATE', sbp_col='SBP', out_col='ShockIndex'):
    # Ensure numeric (in case dtype got set to object somewhere)
    df.loc[:, pulse_col] = pd.to_numeric(df[pulse_col], errors='coerce')
    df.loc[:, sbp_col]   = pd.to_numeric(df[sbp_col],   errors='coerce')

    # ShockIndex = 2.0 if SBP == 0, else PULSERATE / SBP
    df.loc[:, out_col] = np.where(df[sbp_col] == 0, 2.0, df[pulse_col] / df[sbp_col])

# After calling your impute_from_train(...):
recompute_shock_index(X_train_cal)
recompute_shock_index(X_val_cal)
recompute_shock_index(X_test)


In [ ]:
##now for one-hot encoding

# Identify categorical columns from X_train only
categorical_column = [c for c in X_train_cal.columns if X_train_cal[c].dtype == np.dtype('O')]

# Apply pd.get_dummies to all data sets
X_train_cal = pd.get_dummies(X_train_cal, columns=categorical_column, sparse=False)
X_test = pd.get_dummies(X_test, columns=categorical_column, sparse=False)
X_val_cal = pd.get_dummies(X_val_cal, columns=categorical_column, sparse=False)

categorical_column

In [ ]:
#verify data appears as intended
X_train_cal.head()

In [ ]:
##verify no missing data in any split dataset
print(X_train_cal.isnull().sum().sum())
print(X_test.isnull().sum().sum())
print(X_val_cal.isnull().sum().sum())

In [ ]:
##final list of training columns
X_train_cal.columns

In [ ]:
##final list of training columns (test)
X_test.columns

In [ ]:
#verify data is intended size
X_test.shape

In [ ]:
##now with data cleaned, take comparison vars and move them to their own dataframe prior to dropping
new_to_drop = ['TRISS_Death', 'ISS_05']

X_train_ISS=pd.DataFrame()
X_train_ISS['ISS']=X_train_cal['ISS_05']

X_cal_ISS=pd.DataFrame()
X_cal_ISS['ISS']=X_val_cal['ISS_05']

X_ISS=pd.DataFrame()
X_ISS['ISS']=X_test['ISS_05']

X_TRISS=pd.DataFrame()
X_TRISS['TRISS']=X_test['TRISS_Death']

In [ ]:
##now drop those comparison vars from the data that will be fed to the model
X_train_cal.drop(columns=new_to_drop, inplace=True)
X_test.drop(columns=new_to_drop, inplace=True)
X_val_cal.drop(columns=new_to_drop, inplace=True)

In [ ]:
##forcing the test dataset to have exactly the same columns, in the same order, as the training/calibration dataset.
X_test = X_test.reindex(columns=X_train_cal.columns)

In [ ]:
##label all vars in the dataset as either float vars or biinary vars
float32_cols = [
    'AGEYEARS', 'SBP', 'PULSERATE', 'TEMPERATURE', 'WEIGHT',
    'TOTALGCS', 'ISS_05', 'NumberOfInjuries', 'TRISS_Death', 'ShockIndex'
]

binary_cols = [
    'IntracranialVascularInjury', 'BrainStemInjury', 'EDH', 'SAH', 'SDH', 'IPH',
    'SkullFx', 'DAI', 'NeckVascularInjury', 'ThoracicVascularInjury',
    'AeroDigestiveInjury', 'CardiacInjury', 'LungInjury', 'AbdominalVascular',
    'RibFx', 'KidneyInjury', 'StomachInjury', 'SpleenInjury',
    'UroGenInternalInjury', 'SCI', 'SpineFx', 'UEAmputation',
    'UEVascularInjury', 'UELongBoneFx', 'LEVascularInjury', 'PelvicFx',
    'LEAmputation', 'PancreasInjury', 'LELongBoneFx', 'LiverInjury',
    'ColorectalInjury', 'SmallBowelInjury', 'Penetrating', 'HemoPneumo'
]

##coerce continuous vars to be float 32 and binary to int8 to save memory
for df in (X_train_cal, X_test):
    for c in float32_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    for c in binary_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype('int8')

In [ ]:
##Next step is to normalize data

scaler=StandardScaler()
#get the parameters of the transform
scaler.fit(X_train_cal)

#normalize the features in the training set
X_train_s_cal = scaler.transform(X_train_cal)
#normalize the features in the test set
print("After train/test split, X_test shape:", X_test.shape)
X_test_s = scaler.transform(X_test)
print("After scaling, X_test_s shape:", X_test_s.shape)
#normalize the features in the cal set
X_val_s_cal = scaler.transform(X_val_cal)

In [ ]:
##combine datasets now that youved used only X_train to develop the scaler to avoid using calibration data for scaling the training data
X_dev_full = pd.concat([X_train_cal, X_val_cal], axis=0)


In [ ]:
##now, fit model with hyperparameters based on other Jupyternotebook optimization
model_best_gb = xgb.XGBClassifier(random_state=0, colsample_bytree=0.6, learning_rate=0.1, max_depth=7, n_estimators=200, subsample=1.0)
model_best_gb.fit(X_train_s_cal, Y_train_cal)

In [ ]:
# Get predicted probabilities for test set (evaluate model)
from sklearn.metrics import roc_curve, auc, precision_recall_curve, recall_score, confusion_matrix

y_prob_gbo_mtp = model_best_gb.predict_proba(X_test_s)[:, 1]

# Compute AUROC on test set
auroc_gbo = roc_auc_score(Y_test, y_prob_gbo_mtp)
print(f"AUROC on the test set: {auroc_gbo}")

In [ ]:
# Calibrate the model on the validation set
calibrated_model = CalibratedClassifierCV(estimator=model_best_gb, method='isotonic', cv='prefit')
calibrated_model.fit(X_val_s_cal, Y_val_cal)

In [ ]:
# Get predicted probabilities for test set (evaluate model)
y_prob_gbo_mtp = calibrated_model.predict_proba(X_test_s)[:, 1]

# Compute AUROC on test set (calibrated)
auroc_gbo = roc_auc_score(Y_test, y_prob_gbo_mtp)
print(f"AUROC on the test set: {auroc_gbo}")

In [ ]:
##save stored probabilities for comparison against other developed models, if interested
comparison_original = pd.DataFrame({
    "row_id": np.arange(len(Y_test)),
    "y_true": np.asarray(Y_test).ravel(),
    "prob_original": np.asarray(y_prob_gbo_mtp).ravel()
})

# # Replace <SAVE_LOCATION> with the actual path inside your Google Drive (e.g., My Drive/FileNameHere), if using google drive/colab.
comparison_original.to_csv(
    "<SAVE_LOCATTON>.csv",
    index=False
)

In [ ]:
# Get predicted probabilities for test set (evaluate model)
y_prob_gbo = calibrated_model.predict_proba(X_test_s)[:, 1]

# Compute AUROC on test set (calibrated)
auroc_gbo = roc_auc_score(Y_test, y_prob_gbo)
print(f"AUROC on the test set: {auroc_gbo}")


# Calculate ROC curve
y_pred_prob_gbo = calibrated_model.predict_proba(X_test_s)[:, 1]
fpr_gbo, tpr_gbo, thresholds = roc_curve(Y_test, y_pred_prob_gbo)

# Calculate the Area Under the ROC Curve (AUC)
roc_auc_gbo = auc(fpr_gbo, tpr_gbo)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_gbo, tpr_gbo, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc_gbo:.3f})')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

# Calculate precision and recall values
precision, recall, thresholds = precision_recall_curve(Y_test, y_prob_gbo_mtp)

# Calculate area under the precision-recall curve
pr_auc = auc(recall, precision)

# Plot the precision-recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'PR curve (AUC={pr_auc:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.show()

# Choose a custom threshold (e.g., 0.3)
##this value yielded the highest value for accuracy
custom_threshold = 0.5

# Apply the custom threshold to make binary predictions
y_pred_custom_threshold = (y_prob_gbo > custom_threshold).astype(int)

num_positive_cases = sum(y_pred_custom_threshold)
num_negative_cases = len(y_pred_custom_threshold) - num_positive_cases
print("Number of Cases Classified as Positive:", num_positive_cases)
print("Number of Cases Classified as Negative:", num_negative_cases)


# Calculate confusion matrix
conf_matrix = confusion_matrix(Y_test, y_pred_custom_threshold)

print('CONFUSION MATRIX')
print('-----------')
print('|',conf_matrix[0,0],'|', conf_matrix[0,1], '|')
print('-----------')
print('|',conf_matrix[1,0],'|', conf_matrix[1,1], '|')

# Calculate false positive rate (FPR) and false negative rate (FNR)

falseNeg = conf_matrix[1, 0]
falsePos = conf_matrix[0, 1]
trueNeg = conf_matrix[0, 0]
truePos = conf_matrix[1, 1]
actualPos = truePos + falseNeg
actualNeg = trueNeg + falsePos
predPos = truePos + falsePos
predNeg = trueNeg + falseNeg


fpr = falsePos / actualNeg
fnr = falseNeg / actualPos
tpr = truePos / actualPos
tnr = trueNeg / actualNeg
accuracy = (truePos + trueNeg) / (actualPos + actualNeg)
ppV = truePos / predPos
npV = trueNeg / predNeg

brier_score = brier_score_loss(Y_test, y_pred_prob_gbo)

print('Actual Positives:', actualPos)
print('Actual Negatives:', actualNeg)
print("False Positive Rate (FPR):", fpr)
print("False Negative Rate (FNR):", fnr)
print("True Positive Rate (TPR) (Recall/Sensitivity):", tpr) ##right
print("True Negative Rate (TNR) (specificity):", tnr)
print('Accuracy:', accuracy) ##right
print('Precision (PPV),', ppV) ##right
print('NPV', npV)
print("Brier Score:", brier_score)

# Convert probabilities to binary labels using the threshold
y_pred_binary10 = (y_prob_gbo > 0.1).astype(int)
y_pred_binary20 = (y_prob_gbo > 0.2).astype(int)
y_pred_binary30 = (y_prob_gbo > 0.3).astype(int)
y_pred_binary40 = (y_prob_gbo > 0.4).astype(int)
y_pred_binary50 = (y_prob_gbo > 0.5).astype(int)
y_pred_binary60 = (y_prob_gbo > 0.6).astype(int)
y_pred_binary70 = (y_prob_gbo > 0.7).astype(int)
y_pred_binary80 = (y_prob_gbo > 0.8).astype(int)
y_pred_binary90 = (y_prob_gbo > 0.9).astype(int)
y_pred_binary = (y_prob_gbo > custom_threshold).astype(int)

# Compute F1 score
f1 = f1_score(Y_test, y_pred_binary)
f1_10 = f1_score(Y_test, y_pred_binary10)
f1_20 = f1_score(Y_test, y_pred_binary20)
f1_30 = f1_score(Y_test, y_pred_binary30)
f1_40 = f1_score(Y_test, y_pred_binary40)
f1_50 = f1_score(Y_test, y_pred_binary50)
f1_60 = f1_score(Y_test, y_pred_binary60)
f1_70 = f1_score(Y_test, y_pred_binary70)
f1_80 = f1_score(Y_test, y_pred_binary80)
f1_90 = f1_score(Y_test, y_pred_binary90)

print("F1 Score:", f1)
print("F1-10 Score:", f1_10)
print("F1-20 Score:", f1_20)
print("F1-30 Score:", f1_30)
print("F1-40 Score:", f1_40)
print("F1-50 Score:", f1_50)
print("F1-60 Score:", f1_60)
print("F1-70 Score:", f1_70)
print("F1-80 Score:", f1_80)
print("F1-90 Score:", f1_90)

In [ ]:
##evaluate TRISS as predictive tool
predicted_prob_triss = X_TRISS['TRISS']  # This should be probabilities (e.g., 0.8, 0.3)
true_label = Y_test['MORTALITY']          # This should be binary labels (e.g., 1 for death, 0 for survival)

# Calculate FPR, TPR, and thresholds
fpr_triss, tpr_triss, thresholds_triss = roc_curve(true_label, predicted_prob_triss)

# Calculate the area under the ROC curve
roc_auc_triss = auc(fpr_triss, tpr_triss)

# Plot the ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_triss, tpr_triss, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc_triss:.3f}')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--',label='Random')  # Dashed diagonal line for random model
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()



# Print the ROC AUC value
print(f'ROC AUC: {roc_auc_triss:.3f}')

In [ ]:
# Now we created an ISS-derived logistic model to evaluate ISS calibration
#Combine training and calibration portions of the development cohort
X_dev_ISS = pd.concat(
    [X_train_ISS, X_cal_ISS],
    ignore_index=True
)

Y_dev = pd.concat(
    [
        pd.Series(np.asarray(Y_train_cal).ravel()),
        pd.Series(np.asarray(Y_val_cal).ravel())
    ],
    ignore_index=True
)

# Verify alignment
print(len(X_dev_ISS), len(Y_dev))
assert len(X_dev_ISS) == len(Y_dev)

# Fit ISS-to-mortality mapping using development data only
lr_iss = LogisticRegression(max_iter=1000)

lr_iss.fit(
    X_dev_ISS[['ISS']].values,
    Y_dev.values
)

# Generate predicted probabilities for the external test cohort
iss_probs = lr_iss.predict_proba(
    np.asarray(X_ISS).reshape(-1, 1)
)[:, 1]

# Confirm external alignment
assert len(iss_probs) == len(Y_test)

In [ ]:
##evaluate ISS as predictive tool

predicted_prob_iss = iss_probs
true_label = Y_test['MORTALITY']   # This should be the binary outcome variable (0 or 1)

# Calculate the FPR, TPR, and thresholds
fpr_iss, tpr_iss, thresholds_iss = roc_curve(true_label, predicted_prob_iss)

# Calculate the area under the ROC curve (AUROC)
roc_auc_iss = auc(fpr_iss, tpr_iss)

# Plot the ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_triss, tpr_triss, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc_iss:.3f}')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--',label='Random')  # Dashed diagonal line for random model
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()

# Print the AUROC value
print(f'ROC AUC: {roc_auc_iss:.3f}')

In [ ]:
# Plot ROC curves of all 3
plt.figure(figsize=(8, 8))
plt.plot(fpr_gbo, tpr_gbo, color='b', lw=2, label=f'ROC curve (area = {roc_auc_gbo:.3f})')
plt.plot(fpr_triss, tpr_triss, color='green', lw=2, label=f'ROC AUC TRISS = {roc_auc_triss:.3f}')
plt.plot(fpr_iss, tpr_iss, color='darkorange', lw=2, label=f'ROC AUC ISS = {roc_auc_iss:.3f}')
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')

In [ ]:
##now we evaluate calibration of all 3 metrics (MLISS, TRISS, ISS)

def bootstrap_calibration_curve(y_true, y_prob, n_bins=10, n_boot=1000, random_state=None):
    """
    1) Compute the original bin-based calibration curve.
    2) Bootstrap the dataset n_boot times, each time recalculating the bin-based
       fraction of positives (prob_true) and storing it.
    3) Return the original curve + 95% CI per bin (based on 2.5 and 97.5 percentiles).
    """
    # -----------------------------
    # Original calibration curve
    # -----------------------------
    # prob_true_orig, prob_pred_orig = calibration_curve(...) does binning internally.
    # But we want to fix n_bins and ensure consistent binning across bootstraps.
    # We'll do a manual binning approach here to keep consistent bin boundaries.

    # Define bin edges (equally spaced from 0 to 1)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    # Digitize predicted probabilities
    bin_indices = np.digitize(y_prob, bin_edges) - 1
    bin_indices[bin_indices == n_bins] = n_bins - 1  # cap any == n_bins to last bin

    # Prepare arrays to hold the original bin stats
    prob_pred_orig = np.zeros(n_bins)
    prob_true_orig = np.zeros(n_bins)
    counts_in_bin = np.zeros(n_bins, dtype=int)

    # Fill in the stats for each bin
    for i in range(n_bins):
        mask = (bin_indices == i)
        counts_in_bin[i] = np.sum(mask)
        if counts_in_bin[i] > 0:
            prob_pred_orig[i] = np.mean(y_prob[mask])   # mean predicted prob in this bin
            prob_true_orig[i] = np.mean(y_true[mask])   # fraction of positives (actual)
        else:
            # If bin is empty, set to NaN
            prob_pred_orig[i] = np.nan
            prob_true_orig[i] = np.nan

    # Remove empty bins (NaN) from the original arrays
    valid_mask = ~np.isnan(prob_pred_orig)
    prob_pred_orig = prob_pred_orig[valid_mask]
    prob_true_orig = prob_true_orig[valid_mask]

    # -----------------------------
    # Bootstrap to get CIs
    # -----------------------------
    rng = np.random.RandomState(random_state) if random_state else np.random

    # We'll store the fraction of positives (prob_true) for each bin in each bootstrap
    # but only for the bins that were valid in the original data
    boot_prob_true = np.zeros((n_boot, sum(valid_mask)))

    n_data = len(y_true)
    data_idx = np.arange(n_data)

    for b in range(n_boot):
        # Sample with replacement
        sample_indices = rng.randint(0, n_data, size=n_data)
        y_true_b = y_true[sample_indices]
        y_prob_b = y_prob[sample_indices]

        # Repeat the binning steps
        bin_indices_b = np.digitize(y_prob_b, bin_edges) - 1
        bin_indices_b[bin_indices_b == n_bins] = n_bins - 1

        prob_true_b = np.zeros(n_bins)
        for i in range(n_bins):
            mask_b = (bin_indices_b == i)
            if np.sum(mask_b) > 0:
                prob_true_b[i] = np.mean(y_true_b[mask_b])
            else:
                prob_true_b[i] = np.nan

        # filter to only valid bins
        prob_true_b = prob_true_b[valid_mask]
        boot_prob_true[b, :] = prob_true_b

    # Compute 2.5th and 97.5th percentile per bin (column-wise)
    lower_ci = np.nanpercentile(boot_prob_true, 2.5, axis=0)
    upper_ci = np.nanpercentile(boot_prob_true, 97.5, axis=0)

    return prob_pred_orig, prob_true_orig, lower_ci, upper_ci

# -----------------------------------------------------------
# Now apply the function
# -----------------------------------------------------------

# === Input arrays ===
y_true = np.array(Y_test)
y_prob_gbo = np.array(y_prob_gbo)       # ML model
y_prob_triss = np.array(X_TRISS['TRISS'])  # or however you're storing TRISS scores
y_prob_iss = np.array(iss_probs)        # or however you're storing ISS scores

# === Helper function to compute sorted calibration data + CIs ===
def get_bootstrap_calibration_data(y_true, y_prob, label, color, n_bins=10, n_boot=1000, random_state=42):
    prob_pred, prob_true, lower_ci, upper_ci = bootstrap_calibration_curve(
        y_true, y_prob, n_bins=n_bins, n_boot=n_boot, random_state=random_state
    )
    sort_idx = np.argsort(prob_pred)
    return {
        "x": prob_pred[sort_idx],
        "y": prob_true[sort_idx],
        "lower": lower_ci[sort_idx],
        "upper": upper_ci[sort_idx],
        "label": label,
        "color": color
    }

# === Get calibration data ===
calib_gbo = get_bootstrap_calibration_data(y_true, y_prob_gbo, label="ML Model", color='b')
calib_triss = get_bootstrap_calibration_data(y_true, y_prob_triss, label="TRISS", color='g')
calib_iss = get_bootstrap_calibration_data(y_true, y_prob_iss, label="ISS", color='darkorange')

# === Compute Brier scores ===
brier_gbo = brier_score_loss(y_true, y_prob_gbo)
brier_triss = brier_score_loss(y_true, y_prob_triss)
brier_iss = brier_score_loss(y_true, y_prob_iss)

print(f"Brier Score - ML Model: {brier_gbo:.4f}")
print(f"Brier Score - TRISS:    {brier_triss:.4f}")
print(f"Brier Score - ISS:      {brier_iss:.4f}")

# === Plot Reliability Diagram ===
plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

for calib in [calib_gbo
              , calib_triss
              , calib_iss
              ]:
    plt.plot(calib["x"], calib["y"], marker='o', label=calib["label"], color=calib["color"])
    plt.fill_between(calib["x"], calib["lower"], calib["upper"], color=calib["color"], alpha=0.2)

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Mortality Rate')
plt.title('Reliability Diagram with 95% CI – All Methods')
plt.legend(loc='best')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.grid(True)
plt.show()

In [ ]:
##Create helper function to derive calibration intercept and slope
def calibration_intercept_slope(
    y_true,
    y_prob,
    n_boot=1000,
    random_state=42,
    bootstrap=True,
    epsilon=1e-6
):
    """
    Estimate calibration intercept and calibration slope using:

        logit(P(Y=1)) = intercept + slope * logit(predicted_probability)

    Ideal calibration:
        intercept = 0
        slope = 1

    Parameters
    ----------
    y_true : array-like
        Binary observed outcomes coded 0/1.

    y_prob : array-like
        Predicted probabilities.

    n_boot : int
        Number of bootstrap samples.

    random_state : int
        Random seed.

    bootstrap : bool
        Whether to calculate percentile bootstrap confidence intervals.

    epsilon : float
        Predicted probabilities are clipped to [epsilon, 1-epsilon]
        to avoid infinite logits at probabilities of exactly 0 or 1.

    Returns
    -------
    dict
        Calibration intercept, slope, model-based CIs, and bootstrap CIs.
    """

    y_true = np.asarray(y_true, dtype=float).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()

    if len(y_true) != len(y_prob):
        raise ValueError("y_true and y_prob must have the same length.")

    if not np.all(np.isin(np.unique(y_true), [0, 1])):
        raise ValueError("y_true must contain only binary values 0 and 1.")

    if np.any(~np.isfinite(y_prob)):
        raise ValueError("y_prob contains NaN or infinite values.")

    # Avoid infinite logits from probabilities equal to 0 or 1.
    y_prob_clipped = np.clip(y_prob, epsilon, 1 - epsilon)

    # Convert predicted probabilities to predicted log odds.
    predicted_logit = np.log(
        y_prob_clipped / (1 - y_prob_clipped)
    )

    # Add intercept column.
    X = sm.add_constant(predicted_logit)

    # Fit logistic recalibration model.
    model = sm.GLM(
        y_true,
        X,
        family=sm.families.Binomial()
    ).fit()

    intercept = model.params[0]
    slope = model.params[1]

    model_ci = model.conf_int(alpha=0.05)

    results = {
        "intercept": intercept,
        "intercept_model_ci_lower": model_ci[0, 0],
        "intercept_model_ci_upper": model_ci[0, 1],
        "slope": slope,
        "slope_model_ci_lower": model_ci[1, 0],
        "slope_model_ci_upper": model_ci[1, 1]
    }

    if bootstrap:
        rng = np.random.default_rng(random_state)
        n = len(y_true)

        bootstrap_intercepts = []
        bootstrap_slopes = []

        for _ in range(n_boot):
            sample_indices = rng.integers(0, n, size=n)

            y_boot = y_true[sample_indices]
            logit_boot = predicted_logit[sample_indices]

            # A logistic model cannot be estimated if a bootstrap
            # sample contains only one outcome class.
            if len(np.unique(y_boot)) < 2:
                continue

            X_boot = sm.add_constant(logit_boot)

            try:
                boot_model = sm.GLM(
                    y_boot,
                    X_boot,
                    family=sm.families.Binomial()
                ).fit(disp=0)

                bootstrap_intercepts.append(boot_model.params[0])
                bootstrap_slopes.append(boot_model.params[1])

            except Exception:
                # Skip rare bootstrap samples with fitting problems.
                continue

        bootstrap_intercepts = np.asarray(bootstrap_intercepts)
        bootstrap_slopes = np.asarray(bootstrap_slopes)

        if len(bootstrap_intercepts) == 0:
            raise RuntimeError(
                "No bootstrap calibration models could be estimated."
            )

        results.update({
            "intercept_boot_ci_lower": np.percentile(
                bootstrap_intercepts, 2.5
            ),
            "intercept_boot_ci_upper": np.percentile(
                bootstrap_intercepts, 97.5
            ),
            "slope_boot_ci_lower": np.percentile(
                bootstrap_slopes, 2.5
            ),
            "slope_boot_ci_upper": np.percentile(
                bootstrap_slopes, 97.5
            ),
            "successful_bootstraps": len(bootstrap_intercepts)
        })

    return results

In [ ]:
# === Compute calibration intercepts and slopes ===

calibration_gbo = calibration_intercept_slope(
    y_true,
    y_prob_gbo,
    n_boot=1000,
    random_state=42
)

calibration_triss = calibration_intercept_slope(
    y_true,
    y_prob_triss,
    n_boot=1000,
    random_state=42
)

calibration_iss = calibration_intercept_slope(
    y_true,
    y_prob_iss,
    n_boot=1000,
    random_state=42
)

In [ ]:
##print above results
def print_calibration_results(name, results):
    print(f"\n{name}")
    print(
        f"Calibration intercept: "
        f"{results['intercept']:.3f} "
        f"(bootstrap 95% CI "
        f"{results['intercept_boot_ci_lower']:.3f} to "
        f"{results['intercept_boot_ci_upper']:.3f})"
    )
    print(
        f"Calibration slope:     "
        f"{results['slope']:.3f} "
        f"(bootstrap 95% CI "
        f"{results['slope_boot_ci_lower']:.3f} to "
        f"{results['slope_boot_ci_upper']:.3f})"
    )


print_calibration_results("ML Model", calibration_gbo)
print_calibration_results("TRISS", calibration_triss)
print_calibration_results("ISS", calibration_iss)

In [ ]:
# Decision Curve analysis
# ========================================

##first define function to calculate net benefit
def net_benefit(y_true, y_prob, thresholds):
    """
    NetBenefit = (TP/N) - (FP/N)*(threshold/(1-threshold))
    """
    N = len(y_true)
    NB = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        TP = np.sum((y_true == 1) & (y_pred == 1))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        if t == 1.0:
            nb_t = 0
        else:
            nb_t = (TP / N) - (FP / N) * (t / (1 - t))
        NB.append(nb_t)
    return NB

# Thresholds for decision curve
decision_thresholds = np.linspace(0.0, 1.0, 101)

# Make sure y_true is an array
y_true_array = np.asarray(Y_test).flatten()

# Net benefit for MLISS
NB_model = net_benefit(y_true_array, y_prob_gbo, decision_thresholds)

# # Net benefit for ISS (ISS predictions: X_ISS.values)
NB_ISS = net_benefit(y_true_array, iss_probs, decision_thresholds)

# Net benefit for TRISS (TRISS predictions: X_TRISS.values)
NB_TRISS = net_benefit(y_true_array, X_TRISS.values.flatten(), decision_thresholds)

# Net benefit for treat all and treat none (baseline strategies)
N = len(Y_test)
prevalence = np.mean(Y_test)  # fraction of positives
treat_all_nb = []
for t in decision_thresholds:
    if t == 1.0:
        treat_all_nb.append(0)
    else:
        treat_all_nb.append(prevalence - (1 - prevalence)*(t/(1-t)))

treat_none_nb = np.zeros_like(decision_thresholds)

# now plot DCA
plt.figure(figsize=(8, 6))
plt.plot(decision_thresholds, NB_model, label='New ML Model', color='b')
plt.plot(decision_thresholds, NB_ISS, label='ISS', color='darkorange')
plt.plot(decision_thresholds, NB_TRISS, label='TRISS', color='g')
plt.plot(decision_thresholds, treat_all_nb, label='Treat All', color='red', linestyle='--')
plt.plot(decision_thresholds, treat_none_nb, label='Treat None', color='grey', linestyle=':')

plt.xlabel('Threshold Probability')
plt.ylabel('Net Benefit')
plt.title('Decision Curve Analysis')
plt.legend(loc='best')
plt.ylim([-0.06, 0.06])
plt.xlim([0, 1.0])
plt.grid(True)
plt.show()

In [ ]:
#generate secondary metrics
# ========================================
#  Confusion Matrix Stats @ threshold=0.5, similar to above derivation
# ========================================
custom_threshold = 0.5
y_pred_custom_threshold = (y_prob_gbo >= custom_threshold).astype(int)

conf_matrix = confusion_matrix(Y_test, y_pred_custom_threshold)
falseNeg, falsePos, trueNeg, truePos = conf_matrix[1, 0], conf_matrix[0, 1], conf_matrix[0, 0], conf_matrix[1, 1]
actualPos, actualNeg = truePos + falseNeg, trueNeg + falsePos
predPos, predNeg = truePos + falsePos, trueNeg + falseNeg

fpr = falsePos / actualNeg if actualNeg else 0
fnr = falseNeg / actualPos if actualPos else 0
tpr = truePos / actualPos if actualPos else 0
tnr = trueNeg / actualNeg if actualNeg else 0
accuracy = (truePos + trueNeg) / (actualPos + actualNeg) if (actualPos + actualNeg) else 0
ppV = truePos / predPos if predPos != 0 else 0
npV = trueNeg / predNeg if predNeg != 0 else 0

brier_score = brier_score_loss(Y_test, y_prob_gbo)

print("\n=== Threshold = 0.50 ===")
print("Confusion Matrix:")
print(conf_matrix)
print(f"False Positive Rate (FPR): {fpr:.3f}")
print(f"False Negative Rate (FNR): {fnr:.3f}")
print(f"True Positive Rate (TPR/Recall): {tpr:.3f}")
print(f"True Negative Rate (TNR/Specificity): {tnr:.3f}")
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision (PPV): {ppV:.3f}")
print(f"Negative Predictive Value (NPV): {npV:.3f}")
print(f"Brier Score: {brier_score:.4f}")

# ========================================
# Find Threshold that Maximizes F1 (0.01 increments)
# ========================================
candidate_thresholds = np.linspace(0, 1, 101)  # from 0.00 to 1.00 in 0.01 steps
best_f1 = -1.0
best_f1_threshold = 0.0

for t in candidate_thresholds:
    y_pred = (y_prob_gbo >= t).astype(int)
    f1_val = f1_score(Y_test, y_pred)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_f1_threshold = t

print(f"\nOptimal threshold for max F1 (nearest hundredth): {best_f1_threshold:.2f}")
print(f"Max F1 Score at that threshold: {best_f1:.3f}")

# ========================================
# Find Threshold for >=90% Specificity (0.01 increments)
# ========================================
tnr_90_thresh = None

for t in candidate_thresholds:
    y_pred = (y_prob_gbo >= t).astype(int)
    cm = confusion_matrix(Y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    actual_neg = tn + fp
    if actual_neg == 0:
        continue

    specificity = tn / actual_neg  # TNR
    if specificity >= 0.90:
        tnr_90_thresh = t
        break  # first threshold that achieves TNR >= 0.90

if tnr_90_thresh is not None:
    print(f"\nThreshold achieving >=90% specificity (nearest hundredth): {tnr_90_thresh:.2f}")
    # Recompute confusion matrix & TPR at that threshold
    y_pred_90sp = (y_prob_gbo >= tnr_90_thresh).astype(int)
    cm_90 = confusion_matrix(Y_test, y_pred_90sp)
    tn_90, fp_90, fn_90, tp_90 = cm_90.ravel()
    tnr_val_90 = tn_90 / (tn_90 + fp_90) if (tn_90 + fp_90) else 0
    tpr_val_90 = tp_90 / (tp_90 + fn_90) if (tp_90 + fn_90) else 0
    print(f"Specificity (TNR) at {tnr_90_thresh:.2f}: {tnr_val_90:.3f}")
    print(f"Sensitivity (TPR) at {tnr_90_thresh:.2f}: {tpr_val_90:.3f}")
else:
    print("\nNo threshold found where TNR >= 0.90 in [0.00, 1.00].")
# ========================================
# Define a Function to Compute Metrics
# ========================================
def compute_metrics(y_true, y_prob, threshold):
    """
    Computes performance metrics at a given threshold.

    Parameters:
    - y_true: True binary labels
    - y_prob: Predicted probabilities for the positive class
    - threshold: Threshold to convert probabilities to binary predictions

    Returns:
    - metrics_dict: Dictionary containing performance metrics
    """
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
    sensitivity = tp / (tp + fn) if (tp + fn) else 0  # TPR
    specificity = tn / (tn + fp) if (tn + fp) else 0  # TNR
    precision = tp / (tp + fp) if (tp + fp) else 0      # PPV
    npv = tn / (tn + fn) if (tn + fn) else 0            # NPV
    f1 = f1_score(y_true, y_pred) if (tp + fp + fn) else 0

    metrics_dict = {
        'Accuracy': accuracy,
        'Sensitivity (TPR)': sensitivity,
        'Specificity (TNR)': specificity,
        'Precision (PPV)': precision,
        'Negative Predictive Value (NPV)': npv,
        'F1 Score': f1
    }

    return metrics_dict

# ========================================
# Create and Print Performance Tables
# ========================================
# Initialize an empty list to store metric dictionaries
performance_metrics = []

# Thresholds to report
thresholds_to_report = {
    'Max F1 Score Threshold': best_f1_threshold,
    '90% Specificity Threshold': tnr_90_thresh
}

# Compute metrics for each threshold
for desc, thresh in thresholds_to_report.items():
    if thresh is not None:
        metrics = compute_metrics(Y_test, y_prob_gbo, thresh)
        metrics['Threshold'] = f"{thresh:.2f}"
        performance_metrics.append(metrics)
    else:
        print(f"\n{desc} is not available as no threshold achieved >=90% specificity.")

# Create a DataFrame
df_performance = pd.DataFrame(performance_metrics)

# Reorder columns for better readability
df_performance = df_performance[['Threshold', 'Accuracy', 'Sensitivity (TPR)',
                                 'Specificity (TNR)', 'Precision (PPV)',
                                 'Negative Predictive Value (NPV)', 'F1 Score']]

# Set descriptive index
df_performance.index = thresholds_to_report.keys()

# Display the tables
print("\n=== Performance Metrics at Selected Thresholds ===")
print(df_performance.to_string(float_format="{:.3f}".format))

# ISS (already probability)
y_pred_iss = (iss_probs.flatten() >= 0.27).astype(int)
f1_iss = f1_score(Y_test, y_pred_iss)

# TRISS (already probability)
y_pred_triss = (X_TRISS.values.flatten() >= 0.27).astype(int)
f1_triss = f1_score(Y_test, y_pred_triss)

# Now you have best threshold and F1 for your ML model
print("\n=== F1 Scores ===")
print(f"F1 Score (Your ML Model @ optimal threshold): {best_f1:.3f}")
print(f"F1 Score (TRISS @ 0.5 threshold): {f1_triss:.3f}")
print(f"F1 Score (ISS Probs @ 0.5 threshold): {f1_iss:.3f}")


In [ ]:
# ======================================
# Find Optimal Thresholds for Max F1 and ~90% Specificity
# ======================================

candidate_thresholds = np.linspace(0, 1, 101)  # from 0.00 to 1.00 in 0.01 steps

# Initialize trackers
best_f1_iss = -1.0
best_f1_threshold_iss = 0.0
spec90_threshold_iss = None
closest_spec90_diff_iss = float("inf")

best_f1_triss = -1.0
best_f1_threshold_triss = 0.0
spec90_threshold_triss = None
closest_spec90_diff_triss = float("inf")

# Function to compute specificity
def get_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# 1. ISS
for t in candidate_thresholds:
    y_pred = (iss_probs.flatten() >= t).astype(int)
    f1_val = f1_score(Y_test, y_pred)
    spec_val = get_specificity(Y_test, y_pred)

    # Max F1
    if f1_val > best_f1_iss:
        best_f1_iss = f1_val
        best_f1_threshold_iss = t

    # Closest to 90% specificity
    if abs(spec_val - 0.90) < closest_spec90_diff_iss:
        closest_spec90_diff_iss = abs(spec_val - 0.90)
        spec90_threshold_iss = t

# 2. TRISS
for t in candidate_thresholds:
    y_pred = (X_TRISS.values.flatten() >= t).astype(int)
    f1_val = f1_score(Y_test, y_pred)
    spec_val = get_specificity(Y_test, y_pred)

    # Max F1
    if f1_val > best_f1_triss:
        best_f1_triss = f1_val
        best_f1_threshold_triss = t

    # Closest to 90% specificity
    if abs(spec_val - 0.90) < closest_spec90_diff_triss:
        closest_spec90_diff_triss = abs(spec_val - 0.90)
        spec90_threshold_triss = t

# ========================================
# Print Results
# ========================================
print("\n=== Optimal Thresholds ===")
print(f"ISS - Best F1 = {best_f1_iss:.3f} at Threshold = {best_f1_threshold_iss:.2f}")
print(f"ISS - ~90% Specificity Threshold = {spec90_threshold_iss:.2f}")

print(f"TRISS - Best F1 = {best_f1_triss:.3f} at Threshold = {best_f1_threshold_triss:.2f}")
print(f"TRISS - ~90% Specificity Threshold = {spec90_threshold_triss:.2f}")


In [ ]:
# ========================================
# Expanded Performance Metrics for ML, ISS, and TRISS at Both Thresholds
# ========================================

# 1. Compute performance metrics at each threshold
performance_metrics_all = []

# ML Model
for desc, thresh in thresholds_to_report.items():
    if thresh is not None:
        metrics = compute_metrics(Y_test, y_prob_gbo, thresh)
        metrics['Threshold'] = f"{thresh:.2f}"
        metrics['Model'] = 'ML Model'
        metrics['Threshold Type'] = desc
        performance_metrics_all.append(metrics)

# ISS Model at both thresholds
# Max F1
metrics_iss_f1 = compute_metrics(Y_test, iss_probs.flatten(), best_f1_threshold_iss)
metrics_iss_f1['Threshold'] = f"{best_f1_threshold_iss:.2f}"
metrics_iss_f1['Model'] = 'ISS'
metrics_iss_f1['Threshold Type'] = 'Max F1 Score Threshold'
performance_metrics_all.append(metrics_iss_f1)

# ~90% Specificity
metrics_iss_spec90 = compute_metrics(Y_test, iss_probs.flatten(), spec90_threshold_iss)
metrics_iss_spec90['Threshold'] = f"{spec90_threshold_iss:.2f}"
metrics_iss_spec90['Model'] = 'ISS'
metrics_iss_spec90['Threshold Type'] = '90% Specificity Threshold'
performance_metrics_all.append(metrics_iss_spec90)

# TRISS Model at both thresholds
# Max F1
metrics_triss_f1 = compute_metrics(Y_test, X_TRISS.values.flatten(), best_f1_threshold_triss)
metrics_triss_f1['Threshold'] = f"{best_f1_threshold_triss:.2f}"
metrics_triss_f1['Model'] = 'TRISS'
metrics_triss_f1['Threshold Type'] = 'Max F1 Score Threshold'
performance_metrics_all.append(metrics_triss_f1)

# ~90% Specificity
metrics_triss_spec90 = compute_metrics(Y_test, X_TRISS.values.flatten(), spec90_threshold_triss)
metrics_triss_spec90['Threshold'] = f"{spec90_threshold_triss:.2f}"
metrics_triss_spec90['Model'] = 'TRISS'
metrics_triss_spec90['Threshold Type'] = '90% Specificity Threshold'
performance_metrics_all.append(metrics_triss_spec90)

# ========================================
# Create and display Combined DataFrame
# ========================================
df_performance_all = pd.DataFrame(performance_metrics_all)

# Reorder columns
df_performance_all = df_performance_all[['Model', 'Threshold Type', 'Threshold', 'Accuracy',
                                         'Sensitivity (TPR)', 'Specificity (TNR)',
                                         'Precision (PPV)', 'Negative Predictive Value (NPV)',
                                         'F1 Score']]

# Display results
print("\n=== Expanded Performance Metrics ===")
print(df_performance_all.to_string(float_format="{:.3f}".format))


In [ ]:
# Brier score for ISS
brier_score_iss = brier_score_loss(Y_test, iss_probs.flatten())
print(f"Brier Score (ISS): {brier_score_iss:.4f}")

# Brier score for TRISS
brier_score_triss = brier_score_loss(Y_test, X_TRISS.values.flatten())
print(f"Brier Score (TRISS): {brier_score_triss:.4f}")

In [ ]:
##now, comput 95% confidence interval based on 1000-sample bootstrapping

# Initialize your classifier (replace with your actual classifier)
classifier = model_best_gb

# Compute the AUROC on the original test set
original_auroc = auroc_gbo



# Number of bootstrap samples
n_bootstrap_samples = 1000

# Initialize an array to store bootstrapped AUROC values
bootstrap_aurocs = np.zeros(n_bootstrap_samples)

# Perform bootstrapping
for i in range(n_bootstrap_samples):
    # Generate a bootstrap sample
    bootstrap_indices = np.random.choice(len(Y_test), len(Y_test), replace=True)
    y_bootstrap = y_prob_gbo_mtp[bootstrap_indices]
    y_true_bootstrap = Y_test.iloc[bootstrap_indices]

    # Compute AUROC on the bootstrap sample
    bootstrap_aurocs[i] = roc_auc_score(y_true_bootstrap, y_bootstrap)

# Calculate the confidence interval
confidence_interval = np.percentile(bootstrap_aurocs, [2.5, 97.5])

print("Original AUROC:", original_auroc)
print("95% Confidence Interval:", confidence_interval)

In [ ]:
# Compute AUROC ISS
auroc_iss = roc_auc_score(Y_test, X_ISS)
print(f"AUROC on the test set: {auroc_iss}")

In [ ]:
# Compute AUROC TRISS
auroc_triss = roc_auc_score(Y_test, X_TRISS)
print(f"AUROC on the test set: {auroc_triss}")

In [ ]:
##now for paired bootstrap testing between MLISS and TRISS and MLISS and ISS
##first lets define a helper function

def paired_bootstrap_auc_test(
    y_true,
    predA,
    predB,
    n_boot=1000,
    alpha=0.05,
    random_state=None
):
    """
    Compare two correlated AUCs (predA vs. predB) via paired bootstrapping,
    also returning 95% CI for each AUC individually.

    Parameters
    ----------
    y_true : array-like of shape (n_samples,)
        Ground truth binary labels (0 or 1).
    predA : array-like of shape (n_samples,)
        Scores (e.g., probabilities) from model/variable A.
    predB : array-like of shape (n_samples,)
        Scores from model/variable B.
    n_boot : int, default=1000
        Number of bootstrap iterations.
    alpha : float, default=0.05
        Significance level (for the (1 - alpha) CIs).
    random_state : int or None
        Random seed for reproducibility.

    Returns
    -------
    results : dict
        Dictionary containing:
        - aucA, aucB: AUC on the original dataset for each method.
        - aucA_ci_lower, aucA_ci_upper: (1 - alpha) CI for A's AUC.
        - aucB_ci_lower, aucB_ci_upper: (1 - alpha) CI for B's AUC.
        - baseline_diff: AUC(A) - AUC(B) on the full dataset (no resampling).
        - mean_diff: Mean difference of AUCs across bootstrap samples.
        - diff_ci_lower, diff_ci_upper: (1 - alpha) CI for AUC difference.
        - p_value: Approx. two-sided p-value for difference in AUC.
    """

    # Convert to NumPy arrays
    y_true = np.asarray(y_true)
    predA = np.asarray(predA)
    predB = np.asarray(predB)

    # Basic checks
    assert len(y_true) == len(predA) == len(predB), "Arrays must all have the same length."
    n = len(y_true)

    # Compute AUC on the full (original) dataset
    aucA = roc_auc_score(y_true, predA)
    aucB = roc_auc_score(y_true, predB)
    baseline_diff = aucA - aucB

    # Random generator
    rng = np.random.default_rng(random_state)

    # Arrays to store bootstrapped AUCs
    aucAs = np.zeros(n_boot)
    aucBs = np.zeros(n_boot)
    diffs = np.zeros(n_boot)

    for i in range(n_boot):
        # 1) Sample n indices with replacement
        sample_idx = rng.integers(0, n, size=n)

        # 2) Create bootstrap samples
        y_samp = y_true[sample_idx]
        A_samp = predA[sample_idx]
        B_samp = predB[sample_idx]

        # 3) Compute AUC for each method
        aucA_samp = roc_auc_score(y_samp, A_samp)
        aucB_samp = roc_auc_score(y_samp, B_samp)

        # 4) Store AUCs and difference
        aucAs[i] = aucA_samp
        aucBs[i] = aucB_samp
        diffs[i] = aucA_samp - aucB_samp

    # Compute individual AUC CIs
    aucA_ci_lower = np.percentile(aucAs, 100 * (alpha / 2))
    aucA_ci_upper = np.percentile(aucAs, 100 * (1 - alpha / 2))

    aucB_ci_lower = np.percentile(aucBs, 100 * (alpha / 2))
    aucB_ci_upper = np.percentile(aucBs, 100 * (1 - alpha / 2))

    # Compute difference CI
    diff_ci_lower = np.percentile(diffs, 100 * (alpha / 2))
    diff_ci_upper = np.percentile(diffs, 100 * (1 - alpha / 2))

    # Approx. two-sided p-value by sign test on "diffs"
    n_neg = np.sum(diffs < 0)
    n_pos = np.sum(diffs > 0)
    p_val = 2.0 * min(n_neg, n_pos) / n_boot
    p_val = min(p_val, 1.0)

    coverage = (1 - alpha) * 100.0  # e.g., 95.0 if alpha=0.05

    return {
        "aucA": aucA,
        "aucB": aucB,
        "aucA_ci_lower": aucA_ci_lower,
        "aucA_ci_upper": aucA_ci_upper,
        "aucB_ci_lower": aucB_ci_lower,
        "aucB_ci_upper": aucB_ci_upper,
        "baseline_diff": baseline_diff,
        "mean_diff": np.mean(diffs),
        "diff_ci_lower": diff_ci_lower,
        "diff_ci_upper": diff_ci_upper,
        "p_value": p_val,
        "coverage": coverage
    }


In [ ]:
#now actually run paired bootstrapping
# Y_test: ground truth labels (0 or 1)
# ML_preds: predictions from your ML model
# ISS_vals, pTRISS_vals, TRISS_vals: numeric scores

# now actually run paired bootstrapping
# Y_test: ground truth labels (0 or 1)
# ML_preds: predictions from your ML model
# ISS_vals, TRISS_vals: numeric scores

pairs = {
    "ISS": X_ISS.values,
    "TRISS": X_TRISS.values,
}

base_seed = 42

for i, (var_name, var_array) in enumerate(pairs.items()):
    results = paired_bootstrap_auc_test(
        y_true=Y_test,
        predA=y_prob_gbo_mtp,
        predB=var_array,
        n_boot=2000,        # or more for higher precision
        alpha=0.025,       # 97.5% CI in this function (Bonferroni for 2 comparisons)
        random_state=base_seed + i
    )
    coverage_str = f"{results['coverage']:.1f}%"

    print(f"--- ML Model vs. {var_name} ---")
    print(f"AUC(ML) = {results['aucA']:.3f}, {coverage_str} CI: "
          f"[{results['aucA_ci_lower']:.3f}, {results['aucA_ci_upper']:.3f}]")
    print(f"AUC({var_name}) = {results['aucB']:.3f}, {coverage_str} CI: "
          f"[{results['aucB_ci_lower']:.3f}, {results['aucB_ci_upper']:.3f}]")
    print(f"AUC diff (ML - {var_name}) = {results['baseline_diff']:.4f}, {coverage_str} CI: "
          f"[{results['diff_ci_lower']:.4f}, {results['diff_ci_upper']:.4f}]")
    print(f"p-value = {results['p_value']:.4f}\n")


In [ ]:
##now lets get 95% cI for PR curve for MLISS

# Number of bootstrap iterations
n_iterations = 1000
# List to store bootstrapped AUPRC values
auprc_scores = []

# Original precision-recall calculation for actual data
precision, recall, thresholds = precision_recall_curve(Y_test, y_prob_gbo_mtp)
pr_auc = auc(recall, precision)

# Bootstrapping process to calculate 95% CI
for i in range(n_iterations):
    # Resample Y_test and y_prob_gbo_mtp with replacement
    Y_test_resampled, y_prob_resampled = resample(Y_test, y_prob_gbo_mtp)

    # Calculate precision and recall for the bootstrap sample
    precision_resampled, recall_resampled, _ = precision_recall_curve(Y_test_resampled, y_prob_resampled)

    # Calculate AUPRC for the resampled data
    pr_auc_resampled = auc(recall_resampled, precision_resampled)

    # Store the AUPRC score
    auprc_scores.append(pr_auc_resampled)

# Calculate the 95% confidence interval (2.5th and 97.5th percentiles)
lower_bound = np.percentile(auprc_scores, 2.5)
upper_bound = np.percentile(auprc_scores, 97.5)

# Plot the original precision-recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'PR curve (AUC={pr_auc:.3f})')

# Add labels and title
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.show()

# Print the original AUPRC and 95% CI
print(f'AUPRC: {pr_auc:.3f}')
print(f'95% CI for AUPRC: [{lower_bound:.3f}, {upper_bound:.3f}]')

In [ ]:
##now lets get 95% cI for PR curve for ISS

# Number of bootstrap iterations
n_iterations = 1000
# List to store bootstrapped AUPRC values
auprc_scores = []

# Original precision-recall calculation for actual data
precision, recall, thresholds = precision_recall_curve(Y_test, iss_probs)
pr_auc = auc(recall, precision)

# Bootstrapping process to calculate 95% CI
for i in range(n_iterations):
    # Resample Y_test and y_prob_gbo_mtp with replacement
    Y_test_resampled, y_prob_resampled = resample(Y_test,iss_probs)

    # Calculate precision and recall for the bootstrap sample
    precision_resampled, recall_resampled, _ = precision_recall_curve(Y_test_resampled, y_prob_resampled)

    # Calculate AUPRC for the resampled data
    pr_auc_resampled = auc(recall_resampled, precision_resampled)

    # Store the AUPRC score
    auprc_scores.append(pr_auc_resampled)

# Calculate the 95% confidence interval (2.5th and 97.5th percentiles)
lower_bound = np.percentile(auprc_scores, 2.5)
upper_bound = np.percentile(auprc_scores, 97.5)

# Plot the original precision-recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'PR curve (AUC={pr_auc:.3f})')

# Add labels and title
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.show()

# Print the original AUPRC and 95% CI
print(f'AUPRC: {pr_auc:.3f}')
print(f'95% CI for AUPRC: [{lower_bound:.3f}, {upper_bound:.3f}]')

In [ ]:
##now lets get 95% cI for PR curve for TRISS

# Number of bootstrap iterations
n_iterations = 1000
# List to store bootstrapped AUPRC values
auprc_scores = []

# Original precision-recall calculation for actual data
precision, recall, thresholds = precision_recall_curve(Y_test, X_TRISS.values)
pr_auc = auc(recall, precision)

# Bootstrapping process to calculate 95% CI
for i in range(n_iterations):
    # Resample Y_test and y_prob_gbo_mtp with replacement
    Y_test_resampled, y_prob_resampled = resample(Y_test,X_TRISS.values)

    # Calculate precision and recall for the bootstrap sample
    precision_resampled, recall_resampled, _ = precision_recall_curve(Y_test_resampled, y_prob_resampled)

    # Calculate AUPRC for the resampled data
    pr_auc_resampled = auc(recall_resampled, precision_resampled)

    # Store the AUPRC score
    auprc_scores.append(pr_auc_resampled)

# Calculate the 95% confidence interval (2.5th and 97.5th percentiles)
lower_bound = np.percentile(auprc_scores, 2.5)
upper_bound = np.percentile(auprc_scores, 97.5)

# Plot the original precision-recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'PR curve (AUC={pr_auc:.3f})')

# Add labels and title
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.show()

# Print the original AUPRC and 95% CI
print(f'AUPRC: {pr_auc:.3f}')
print(f'95% CI for AUPRC: [{lower_bound:.3f}, {upper_bound:.3f}]')

In [ ]:
##Develop helper function to generate net reclassification improvement
def categorical_nri(
    y_true,
    prob_new,
    prob_reference,
    thresholds
):
    """
    Calculate categorical Net Reclassification Improvement.

    Parameters
    ----------
    y_true : array-like
        Binary outcomes: 1 = death, 0 = survival.

    prob_new : array-like
        Predicted mortality probabilities from the new model, ML-ISS.

    prob_reference : array-like
        Predicted mortality probabilities from the reference model.

    thresholds : sequence of floats
        Prespecified risk-category boundaries.

        Example:
            [0.05, 0.10, 0.25]

        This creates:
            <5%
            5% to <10%
            10% to <25%
            >=25%

    Returns
    -------
    dict
        Event, nonevent, and total NRI components, plus movement counts.
    """

    y_true = np.asarray(y_true).ravel().astype(int)
    prob_new = np.asarray(prob_new).ravel()
    prob_reference = np.asarray(prob_reference).ravel()

    if not (
        len(y_true)
        == len(prob_new)
        == len(prob_reference)
    ):
        raise ValueError("All arrays must have the same length.")

    if not np.all(np.isin(np.unique(y_true), [0, 1])):
        raise ValueError("y_true must contain only 0 and 1.")

    thresholds = np.asarray(thresholds, dtype=float)

    if np.any(np.diff(thresholds) <= 0):
        raise ValueError("thresholds must be strictly increasing.")

    if np.any((thresholds <= 0) | (thresholds >= 1)):
        raise ValueError("thresholds must be between 0 and 1.")

    # Category numbers: 0, 1, 2, ..., number of thresholds.
    category_new = np.digitize(prob_new, thresholds)
    category_reference = np.digitize(prob_reference, thresholds)

    movement = category_new - category_reference

    event_mask = y_true == 1
    nonevent_mask = y_true == 0

    n_events = event_mask.sum()
    n_nonevents = nonevent_mask.sum()

    if n_events == 0 or n_nonevents == 0:
        raise ValueError(
            "Both deaths and survivors are required."
        )

    # Among deaths, moving upward is correct.
    event_up = np.sum(movement[event_mask] > 0)
    event_down = np.sum(movement[event_mask] < 0)

    event_up_rate = event_up / n_events
    event_down_rate = event_down / n_events

    nri_events = event_up_rate - event_down_rate

    # Among survivors, moving downward is correct.
    nonevent_down = np.sum(movement[nonevent_mask] < 0)
    nonevent_up = np.sum(movement[nonevent_mask] > 0)

    nonevent_down_rate = nonevent_down / n_nonevents
    nonevent_up_rate = nonevent_up / n_nonevents

    nri_nonevents = nonevent_down_rate - nonevent_up_rate

    nri_total = nri_events + nri_nonevents

    return {
        "nri_events": nri_events,
        "nri_nonevents": nri_nonevents,
        "nri_total": nri_total,
        "event_up_n": event_up,
        "event_down_n": event_down,
        "event_unchanged_n": (
            n_events - event_up - event_down
        ),
        "nonevent_down_n": nonevent_down,
        "nonevent_up_n": nonevent_up,
        "nonevent_unchanged_n": (
            n_nonevents - nonevent_down - nonevent_up
        ),
        "n_events": n_events,
        "n_nonevents": n_nonevents
    }

In [ ]:
##And now a helper function to generate paired bootstrap confidence intervals for the NRI
def bootstrap_categorical_nri(
    y_true,
    prob_new,
    prob_reference,
    thresholds,
    n_boot=2000,
    random_state=42
):
    """
    Paired bootstrap confidence intervals for categorical NRI.
    """

    y_true = np.asarray(y_true).ravel().astype(int)
    prob_new = np.asarray(prob_new).ravel()
    prob_reference = np.asarray(prob_reference).ravel()

    observed = categorical_nri(
        y_true=y_true,
        prob_new=prob_new,
        prob_reference=prob_reference,
        thresholds=thresholds
    )

    rng = np.random.default_rng(random_state)
    n = len(y_true)

    boot_event = []
    boot_nonevent = []
    boot_total = []

    for _ in range(n_boot):
        indices = rng.integers(0, n, size=n)

        y_b = y_true[indices]

        # Skip samples lacking one outcome group.
        if len(np.unique(y_b)) < 2:
            continue

        try:
            nri_b = categorical_nri(
                y_true=y_b,
                prob_new=prob_new[indices],
                prob_reference=prob_reference[indices],
                thresholds=thresholds
            )

            boot_event.append(nri_b["nri_events"])
            boot_nonevent.append(nri_b["nri_nonevents"])
            boot_total.append(nri_b["nri_total"])

        except ValueError:
            continue

    boot_event = np.asarray(boot_event)
    boot_nonevent = np.asarray(boot_nonevent)
    boot_total = np.asarray(boot_total)

    if len(boot_total) == 0:
        raise RuntimeError(
            "No bootstrap NRI estimates were generated."
        )

    results = observed.copy()

    results.update({
        "nri_events_ci_lower":
            np.percentile(boot_event, 2.5),
        "nri_events_ci_upper":
            np.percentile(boot_event, 97.5),

        "nri_nonevents_ci_lower":
            np.percentile(boot_nonevent, 2.5),
        "nri_nonevents_ci_upper":
            np.percentile(boot_nonevent, 97.5),

        "nri_total_ci_lower":
            np.percentile(boot_total, 2.5),
        "nri_total_ci_upper":
            np.percentile(boot_total, 97.5),

        "successful_bootstraps": len(boot_total)
    })

    return results

In [ ]:
##establish thresholds and run MLISS vs TRISS
risk_thresholds = [0.02, .05, 0.10, 0.25, 0.50]

nri_ml_vs_triss = bootstrap_categorical_nri(
    y_true=np.asarray(Y_test).ravel(),
    prob_new=np.asarray(y_prob_gbo_mtp).ravel(),
    prob_reference=np.asarray(y_prob_triss).ravel(),
    thresholds=risk_thresholds,
    n_boot=2000,
    random_state=42
)

In [ ]:
##run MLISS vs ISS
nri_ml_vs_iss = bootstrap_categorical_nri(
    y_true=np.asarray(Y_test).ravel(),
    prob_new=np.asarray(y_prob_gbo_mtp).ravel(),
    prob_reference=np.asarray(iss_probs).ravel(),
    thresholds=risk_thresholds,
    n_boot=2000,
    random_state=42
)

In [ ]:
##print results for NRI
def print_nri(name, results):
    print(f"\n{name}")

    print(
        "Event NRI: "
        f"{results['nri_events']:.3f} "
        f"(95% CI "
        f"{results['nri_events_ci_lower']:.3f} to "
        f"{results['nri_events_ci_upper']:.3f})"
    )

    print(
        "Nonevent NRI: "
        f"{results['nri_nonevents']:.3f} "
        f"(95% CI "
        f"{results['nri_nonevents_ci_lower']:.3f} to "
        f"{results['nri_nonevents_ci_upper']:.3f})"
    )

    print(
        "Total NRI: "
        f"{results['nri_total']:.3f} "
        f"(95% CI "
        f"{results['nri_total_ci_lower']:.3f} to "
        f"{results['nri_total_ci_upper']:.3f})"
    )


print_nri(
    "ML-ISS versus TRISS",
    nri_ml_vs_triss
)

print_nri(
    "ML-ISS versus ISS",
    nri_ml_vs_iss
)